# Carga de datos

En este notebook realizamos la carga inicial de la base de datos del Proyecto Integrador del Módulo 5. El objetivo es comprobar que el archivo puede leerse correctamente, revisar su estructura general e identificar condiciones iniciales relevantes antes de continuar con el análisis exploratorio y la ingeniería de características.

Esta etapa no modifica los datos. Su función es validar la lectura del archivo, conocer sus dimensiones, revisar columnas disponibles, detectar valores nulos y duplicados, y clasificar las variables de forma preliminar.

In [41]:
import pandas as pd
from pathlib import Path
from IPython.display import display

## 1. Lectura del archivo

Primero definimos la ruta del archivo `Base_de_datos.csv`. El notebook está ubicado dentro de `src/`, por lo que la ruta principal esperada es `../Base_de_datos.csv`. También se incluye una ruta alternativa para poder ejecutar el notebook desde la raíz del proyecto sin modificar el código.

In [42]:
rutas_posibles = [
    Path("../Base_de_datos.csv"),
    Path("Base_de_datos.csv"),
]

ruta_csv = None

for ruta in rutas_posibles:
    if ruta.exists():
        ruta_csv = ruta
        break

if ruta_csv is None:
    raise FileNotFoundError("No se encontró Base_de_datos.csv en la raíz del proyecto ni en la ruta superior.")

print(f"Archivo cargado desde: {ruta_csv}")

df = pd.read_csv(ruta_csv)
display(df.head())

Archivo cargado desde: ..\Base_de_datos.csv


,tipo_credito,fecha_prestamo,capital_prestado,plazo_meses,edad_cliente,tipo_laboral,salario_cliente,total_otros_prestamos,cuota_pactada,puntaje,...,saldo_mora,saldo_total,saldo_principal,saldo_mora_codeudor,creditos_sectorFinanciero,creditos_sectorCooperativo,creditos_sectorReal,promedio_ingresos_datacredito,tendencia_ingresos,Pago_atiempo
0,7,2024-12-21 11:31:35,"3,692,160.00",10,42,Independiente,8000000,2500000,341296,88.77,...,0.00,"51,258.00","51,258.00",0.00,5,0,0,"908,526.00",Estable,1
1,4,2025-04-22 09:47:35,"840,000.00",6,60,Empleado,3000000,2000000,124876,95.23,...,0.00,"8,673.00","8,673.00",0.00,0,0,2,"939,017.00",Creciente,1
2,9,2026-01-08 12:22:40,"5,974,028.40",10,36,Independiente,4036000,829000,529554,47.61,...,0.00,"18,702.00","18,702.00",0.00,3,0,0,NaN,NaN,0
3,4,2025-08-04 12:04:10,"1,671,240.00",6,48,Empleado,1524547,498000,252420,95.23,...,0.00,"15,782.00","15,782.00",0.00,3,0,0,"1,536,193.00",Creciente,1
4,9,2025-04-26 11:24:26,"2,781,636.00",11,44,Empleado,5000000,4000000,217037,95.23,...,0.00,"204,804.00","204,804.00",0.00,3,0,1,"933,473.00",Creciente,1


Después de cargar la base, revisamos cuántos registros y variables contiene el dataset. Esto permite tener una primera idea del tamaño de la información disponible para el análisis y el modelado posterior.

In [43]:
filas, columnas = df.shape
print(f"Filas: {filas}")
print(f"Columnas: {columnas}")

Filas: 10763
Columnas: 23


También revisamos el nombre de las columnas para identificar las variables disponibles y confirmar la presencia de la variable objetivo del proyecto.

In [44]:
columnas_df = pd.DataFrame({
    "columna": df.columns,
    "tipo_dato": [str(tipo) for tipo in df.dtypes.values]
})

display(columnas_df)

,columna,tipo_dato
0,tipo_credito,int64
1,fecha_prestamo,str
2,capital_prestado,float64
3,plazo_meses,int64
4,edad_cliente,int64
5,tipo_laboral,str
6,salario_cliente,int64
7,total_otros_prestamos,int64
8,cuota_pactada,int64
9,puntaje,float64


## 2. Información general del dataset

Ahora revisamos la estructura general de la base de datos: tipos de variables, cantidad de valores no nulos y composición inicial del DataFrame. Esta revisión permite detectar posibles problemas antes de avanzar al análisis exploratorio detallado.

In [45]:
resumen_df = pd.DataFrame({
    "columna": df.columns,
    "tipo_dato": df.dtypes.astype(str).values,
    "no_nulos": df.notnull().sum().values,
    "nulos": df.isnull().sum().values,
    "porcentaje_nulos": (df.isnull().sum().values / len(df) * 100).round(2)
})

display(resumen_df)

,columna,tipo_dato,no_nulos,nulos,porcentaje_nulos
0,tipo_credito,int64,10763,0,0.00
1,fecha_prestamo,str,10763,0,0.00
2,capital_prestado,float64,10763,0,0.00
3,plazo_meses,int64,10763,0,0.00
4,edad_cliente,int64,10763,0,0.00
5,tipo_laboral,str,10763,0,0.00
6,salario_cliente,int64,10763,0,0.00
7,total_otros_prestamos,int64,10763,0,0.00
8,cuota_pactada,int64,10763,0,0.00
9,puntaje,float64,10763,0,0.00


#### Registros nulos

Los valores nulos pueden afectar el análisis exploratorio y el entrenamiento de modelos, por lo que es necesario identificarlos desde la etapa inicial de carga de datos.

In [46]:
nulos_df = df.isnull().sum().reset_index()
nulos_df.columns = ["columna", "nulos"]
nulos_df["porcentaje"] = (nulos_df["nulos"] / len(df) * 100).round(2)

nulos_df = nulos_df[nulos_df["nulos"] > 0].sort_values(
    by="nulos",
    ascending=False
)

if nulos_df.empty:
    print("No se identificaron valores nulos en el dataset.")
else:
    display(nulos_df)

,columna,nulos,porcentaje
21,tendencia_ingresos,2932,27.24
20,promedio_ingresos_datacredito,2930,27.22
16,saldo_mora_codeudor,590,5.48
15,saldo_principal,405,3.76
13,saldo_mora,156,1.45
14,saldo_total,156,1.45
10,puntaje_datacredito,6,0.06


De acuerdo con la revisión de valores nulos realizada anteriormente, algunas variables presentan datos faltantes en proporciones relevantes. Estas columnas deberán ser consideradas en la etapa de ingeniería de características para definir un tratamiento adecuado.

Por ahora no se modifican los valores faltantes, ya que el objetivo de este notebook es validar la carga de datos y comprender su estado inicial antes de aplicar transformaciones.

#### Registros duplicados

También revisamos si existen registros duplicados completos dentro de la base, ya que podrían alterar los resultados del análisis o sesgar el entrenamiento del modelo.

In [47]:
duplicados = df.duplicated().sum()
print(f"Registros duplicados completos: {duplicados}")

Registros duplicados completos: 0


No se identificaron registros duplicados completos en la base de datos. Esto indica que, al menos a nivel de filas completas, no hay repeticiones que deban eliminarse antes del análisis exploratorio.

## 3. Clasificación inicial de variables

Como parte de la carga inicial, separamos las columnas según su tipo de dato. Esto permite distinguir entre variables numéricas, categóricas y de fecha, lo cual será útil para el análisis exploratorio y la ingeniería de características.

In [48]:
from pandas.api.types import (
    is_numeric_dtype,
    is_datetime64_any_dtype,
    is_object_dtype,
    is_string_dtype
)

df["fecha_prestamo"] = pd.to_datetime(
    df["fecha_prestamo"],
    errors="coerce"
)

columnas_numericas = [
    col for col in df.columns
    if is_numeric_dtype(df[col])
]

columnas_fecha = [
    col for col in df.columns
    if is_datetime64_any_dtype(df[col])
]

columnas_categoricas = [
    col for col in df.columns
    if (is_object_dtype(df[col]) or is_string_dtype(df[col]))
    and col not in columnas_fecha
]

clasificacion_variables = pd.DataFrame({
    "tipo_variable": ["numérica", "categórica", "fecha"],
    "cantidad": [
        len(columnas_numericas),
        len(columnas_categoricas),
        len(columnas_fecha)
    ],
    "columnas": [
        ", ".join(columnas_numericas),
        ", ".join(columnas_categoricas),
        ", ".join(columnas_fecha)
    ]
})

display(clasificacion_variables)

,tipo_variable,cantidad,columnas
0,numérica,20,"tipo_credito, capital_prestado, plazo_meses, e..."
1,categórica,2,"tipo_laboral, tendencia_ingresos"
2,fecha,1,fecha_prestamo


La variable objetivo `Pago_atiempo` se identifica dentro del conjunto de datos, pero su análisis detallado se desarrolla en el notebook `comprension_eda.ipynb`, donde corresponde realizar la exploración de la variable objetivo y sus relaciones con el resto de variables.

#### Descripción estadística inicial

Finalmente, obtenemos una primera descripción estadística de las variables numéricas. Esta revisión no sustituye al EDA, pero permite observar rangos, promedios y posibles valores extremos de manera preliminar.

In [49]:
display(df.describe().T)
pd.set_option("display.float_format", "{:,.2f}".format)

,count,mean,min,25%,50%,75%,max,std
tipo_credito,"10,763.00",5.41,4.00,4.00,4.00,9.00,68.00,2.34
fecha_prestamo,10763,2025-04-16 23:06:02.111121,2024-11-26 09:17:04,2025-01-20 17:33:07.500000,2025-03-27 16:23:12,2025-06-16 13:27:58,2026-04-26 18:43:52,NaN
capital_prestado,"10,763.00","2,434,315.00","360,000.00","1,224,831.00","1,921,920.00","3,084,840.00","41,444,152.80","1,909,642.76"
plazo_meses,"10,763.00",10.58,2.00,6.00,10.00,12.00,90.00,6.63
edad_cliente,"10,763.00",43.95,19.00,33.00,42.00,53.00,123.00,15.06
salario_cliente,"10,763.00","17,216,431.46",0.00,"2,000,000.00","3,000,000.00","4,875,808.00","22,000,000,000.00","355,476,717.60"
total_otros_prestamos,"10,763.00","6,238,869.65",0.00,"500,000.00","1,000,000.00","2,000,000.00","6,787,675,263.00","118,418,316.94"
cuota_pactada,"10,763.00","243,617.41","23,944.00","121,041.50","182,863.00","287,833.50","3,816,752.00","210,493.69"
puntaje,"10,763.00",91.17,-38.01,95.23,95.23,95.23,95.23,16.47
puntaje_datacredito,"10,757.00",780.79,-7.00,757.00,791.00,825.00,999.00,104.88


## 4. Conclusiones preliminares

Con esta primera revisión identificamos las dimensiones del dataset, las columnas disponibles, los tipos de datos, la presencia de valores nulos y la ausencia de registros duplicados.

Este notebook se mantiene enfocado en la carga y validación inicial de los datos. El análisis exploratorio detallado, incluyendo la distribución de la variable objetivo y sus relaciones con otras variables, se desarrolla posteriormente en `comprension_eda.ipynb`.

Con esta validación inicial, el proyecto queda preparado para continuar con la etapa de comprensión exploratoria de datos, ingeniería de características y modelamiento supervisado.